In [1]:
import sys
sys.path.append("..")
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch
from src.models import *
from src.data.evaluation import EvalutationDataset
from torch.utils.data import DataLoader
dataloader = DataLoader(EvalutationDataset(root="../data/raw", train=False), batch_size=1, shuffle=False)

regression = Unet_Regression()
regression.load_state_dict(torch.load("../checkpoints/models/Unet_regression/unet_regression_epoch_500.pth"))
classification = DeepCNN()
classification.load_state_dict(torch.load("../checkpoints/models/Zhang/zhang_epoch_500.pth")['model'])
GAN = UNetGenerator()
GAN.load_state_dict(torch.load("../checkpoints/models/GAN/gan_epoch_500.pth")['G'])

Files already downloaded and verified


<All keys matched successfully>

In [3]:
from src.color import *
from src.evalutaion import *

In [4]:
#========= Regression ==========
regression.eval()
regression_psnr, regression_ssim, regression_clfn= 0.0, 0.0, 0.0
for img, L, ab, label in dataloader:
    img = img.squeeze(0).numpy()
    with torch.no_grad():
        regression_output = regression(L)
    regression_output = lab_to_rgb(L, regression_output).squeeze(0) * 255.0
    psnr_regression = calculate_psnr(img, regression_output)
    ssim_regression = calculate_ssim(img, regression_output)
    clfn_regression = calculate_colorfulness(regression_output)
    regression_psnr += psnr_regression
    regression_ssim += ssim_regression
    regression_clfn += clfn_regression
regression_psnr /= len(dataloader)
regression_ssim /= len(dataloader)
regression_clfn /= len(dataloader)
print(f"Regression PSNR: {regression_psnr:.6f}")
print(f"Regression SSIM: {regression_ssim:.6f}")
print(f"Regression Colorfulness: {regression_clfn:.6f}")

Regression PSNR: 24.818386
Regression SSIM: 0.997723
Regression Colorfulness: 0.594146


In [5]:
#========= Classification ==========
classification.eval()
classification_psnr, classification_ssim, classification_clfn = 0.0, 0.0, 0.0
for img, L, ab, label in dataloader:
    img = img.squeeze(0).numpy()
    with torch.no_grad():
        classification_output = classification(L)
    classification_output = lab_to_rgb(L, soft_decode(classification_output, T=0.38)).squeeze(0) * 255.0
    psnr_classification = calculate_psnr(img, classification_output)
    ssim_classification = calculate_ssim(img, classification_output)
    clfn_classification = calculate_colorfulness(classification_output)
    classification_psnr += psnr_classification
    classification_ssim += ssim_classification
    classification_clfn += clfn_classification
classification_psnr /= len(dataloader)
classification_ssim /= len(dataloader)
classification_clfn /= len(dataloader)
print(f"Classification PSNR: {classification_psnr:.6f}")
print(f"Classification SSIM: {classification_ssim:.6f}")
print(f"Classification Colorfulness: {classification_clfn:.6f}")

Classification PSNR: 22.420557
Classification SSIM: 0.996775
Classification Colorfulness: 3.542017


In [6]:
#========= GAN ==========
GAN.eval()
GAN_psnr, GAN_ssim, GAN_clfn = 0.0, 0.0, 0.0
for img, L, ab, label in dataloader:
    img = img.squeeze(0).numpy()
    with torch.no_grad():
        GAN_output = GAN(L)
    GAN_output = lab_to_rgb(L, GAN_output).squeeze(0) * 255.0
    psnr_GAN = calculate_psnr(img, GAN_output)
    ssim_GAN = calculate_ssim(img, GAN_output)
    clfn_GAN = calculate_colorfulness(GAN_output)
    GAN_psnr += psnr_GAN
    GAN_ssim += ssim_GAN
    GAN_clfn += clfn_GAN
GAN_psnr /= len(dataloader)
GAN_ssim /= len(dataloader)
GAN_clfn /= len(dataloader)
print(f"GAN PSNR: {GAN_psnr:.6f}")
print(f"GAN SSIM: {GAN_ssim:.6f}")
print(f"GAN Colorfulness: {GAN_clfn:.6f}")

GAN PSNR: 23.762046
GAN SSIM: 0.997187
GAN Colorfulness: 0.912475
